# 03. Graph + Vector — GraphRAG

Apache AGE 그래프와 pgvector 벡터 검색을 하나의 PostgreSQL에서 결합.

- 그래프 노드를 자동으로 벡터화
- 벡터 검색 결과에 그래프 컨텍스트 연결
- LLM 기반 Cypher QA 체인

**사전 준비**: Docker 컨테이너 + `pip install "langchain-age[all]"`

In [ ]:
import hashlib

DSN = "host=localhost port=5433 dbname=langchain_age user=langchain password=langchain"

class DemoEmbeddings:
    """API 키 없이 동작하는 결정적 임베딩 (dim=64)."""
    def embed_documents(self, texts):
        return [self._hash_embed(t) for t in texts]
    def embed_query(self, text):
        return self._hash_embed(text)
    def _hash_embed(self, text, dim=64):
        h = hashlib.sha256(text.encode()).digest()
        return [b / 255.0 for b in h * (dim // len(h) + 1)][:dim]

embeddings = DemoEmbeddings()

## 1. 그래프에 지식 구축

In [ ]:
from langchain_age import AGEGraph

graph = AGEGraph(DSN, graph_name="techkg")

# 기술 지식 그래프 구축
data = [
    ("PostgreSQL", "Database", "Open-source relational database with extensibility"),
    ("Apache AGE", "Extension", "Graph query extension for PostgreSQL using Cypher"),
    ("pgvector", "Extension", "Vector similarity search extension for PostgreSQL"),
    ("LangChain", "Framework", "Framework for building applications with LLMs"),
    ("Neo4j", "Database", "Native graph database with Bolt protocol"),
    ("Python", "Language", "General-purpose programming language for AI and data science"),
]

for name, label, description in data:
    graph.query(
        f"MERGE (n:{label} {{name: '{name}', description: '{description}'}})"
    )

# 관계
rels = [
    ("Apache AGE", "Extension", "PostgreSQL", "Database", "EXTENDS"),
    ("pgvector", "Extension", "PostgreSQL", "Database", "EXTENDS"),
    ("LangChain", "Framework", "Python", "Language", "BUILT_WITH"),
    ("LangChain", "Framework", "Neo4j", "Database", "INTEGRATES_WITH"),
    ("LangChain", "Framework", "PostgreSQL", "Database", "INTEGRATES_WITH"),
]

for src, src_l, tgt, tgt_l, rel_type in rels:
    graph.query(
        f"MATCH (a:{src_l} {{name: '{src}'}}), (b:{tgt_l} {{name: '{tgt}'}}) "
        f"MERGE (a)-[:{rel_type}]->(b)"
    )

graph.refresh_schema()
print(graph.schema)

## 2. 그래프 노드 → 벡터화 (`from_existing_graph`)

그래프에 이미 저장된 노드의 텍스트 프로퍼티를 임베딩하여 벡터 테이블 생성.

In [ ]:
from langchain_age import AGEVector

# Database 타입 노드를 벡터화
db_store = AGEVector.from_existing_graph(
    embedding=embeddings,
    connection_string=DSN,
    graph_name="techkg",
    node_label="Database",
    text_node_properties=["name", "description"],
    collection_name="techkg_databases",
)

# Extension 타입 노드를 벡터화
ext_store = AGEVector.from_existing_graph(
    embedding=embeddings,
    connection_string=DSN,
    graph_name="techkg",
    node_label="Extension",
    text_node_properties=["name", "description"],
    collection_name="techkg_extensions",
)

print(f"Databases vectorised: {len(db_store.similarity_search('x', k=100))}")
print(f"Extensions vectorised: {len(ext_store.similarity_search('x', k=100))}")

## 3. 벡터 검색 → 그래프 컨텍스트 확장

벡터 검색으로 관련 노드를 찾고, 그래프에서 추가 컨텍스트를 가져오는 패턴.

In [ ]:
# Step 1: 벡터 검색으로 관련 데이터베이스 찾기
results = db_store.similarity_search("graph database", k=2)
print("Vector search results:")
for doc in results:
    print(f"  - {doc.page_content}")
    print(f"    metadata: {doc.metadata}")

# Step 2: 그래프에서 해당 노드의 관계 확장
for doc in results:
    label = doc.metadata.get("node_label", "")
    # 노드의 name 추출 (page_content의 첫 단어)
    name = doc.page_content.split()[0]
    neighbors = graph.query(
        f"MATCH (n:{label} {{name: '{name}'}})-[r]->(m) "
        f"RETURN type(r) AS rel, m.name AS neighbor LIMIT 5"
    )
    if neighbors:
        print(f"  Graph context for {name}:")
        for n in neighbors:
            print(f"    -{n['rel']}-> {n['neighbor']}")

## 4. QA Chain (LLM이 있는 경우)

LLM이 Cypher를 생성하고, AGE에서 실행한 결과로 답변.

In [ ]:
# OpenAI API 키가 있을 때만 실행
import os
if os.getenv("OPENAI_API_KEY"):
    from langchain_openai import ChatOpenAI
    from langchain_age import AGEGraphCypherQAChain

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    chain = AGEGraphCypherQAChain.from_llm(
        llm,
        graph=graph,
        allow_dangerous_requests=True,
        return_intermediate_steps=True,
    )

    result = chain.invoke({"query": "What extensions does PostgreSQL have?"})
    print(f"Answer: {result['result']}")
    print(f"Cypher: {result['intermediate_steps'][0]['query']}")
else:
    print("OPENAI_API_KEY not set — skipping QA chain demo.")
    print("Set it with: export OPENAI_API_KEY=sk-...")

## 5. LangGraph Store 연동

같은 PostgreSQL 인스턴스에서 `langgraph-checkpoint-postgres`의 `PostgresStore`를 사용하여 장기 메모리를 관리할 수 있다. AGE 그래프, pgvector 테이블, LangGraph store 테이블이 동일한 DB에 공존.

In [ ]:
# langgraph-checkpoint-postgres가 설치되어 있을 때만 실행
try:
    from langgraph.store.postgres import PostgresStore

    # 같은 DB에 연결 — AGE/pgvector 테이블과 공존
    with PostgresStore.from_conn_string(DSN) as store:
        store.setup()
        store.put(("demo",), "key1", {"note": "This coexists with AGE graph data"})
        item = store.get(("demo",), "key1")
        print(f"LangGraph Store: {item.value}")
except ImportError:
    print("langgraph-checkpoint-postgres not installed — skipping.")
    print("Install: pip install langgraph-checkpoint-postgres")

## 6. 정리

In [ ]:
db_store._drop_table()
ext_store._drop_table()
db_store.close()
ext_store.close()
# graph.drop_graph()  # 그래프도 삭제하려면 주석 해제
graph.close()
print("Done.")